# Demo of all Tools!

This notebook walks through a full demonstration of tools to use on Jubilee with a lab automation deck:
1. **Jogging** — move the machine around 
2. **Serial Dilution** — use the syringe to prepare a color gradient in a 24-well plate
3. **Pipette Transfer** — use the P300 pipette to move a precise volume from the last dilution well (A3) to A4
4. **Camera** — photograph one well from the dilution series
5. **Spectral Sensor** — measure the spectrum of the same well and print the results

Note that this notebook assumes some familiarity of all tools and is a 'refresher' notebook; dive into the other example notebooks if you want more details on how the tools are used! This notebook can only be run on Jubilees with a lab automation deck; if your maching doesn't have one, check out the tool examples in the example folder.

### Before Starting
- Clear any existing items off the bed!
- Make sure the syringe plunger is at 0 mL before powering on (see syringe intro notebook).
- Have ready: 2 petri dishes (one with water, one with colored water) and a 24-well plate.

In [ ]:
# Import everything we'll need for this workflow
from science_jubilee.Machine import Machine
from science_jubilee.tools.Syringe import Syringe
from science_jubilee.tools.Pipette import Pipette
from science_jubilee.tools.Camera import Camera
from science_jubilee.tools.AS7341 import AS7341

In [ ]:
# Connect to the machine
m = Machine(address='192.168.1.2')

In [ ]:
# m.home_all()  # uncomment if the machine needs to be homed

---
## Part 1: Jogging the Machine

Use `move_to` for absolute positions and `move` for relative moves.
See the [start here](./examples/start-here/readme.md) folder for further details

In [ ]:
def mm_sec(speed):
    """Convert mm/sec to mm/min for use with move_to/move."""
    return speed * 60.0

In [ ]:
# Relative moves — drop the bed
m.move(dz=15)           # drop bed for clearance
# Absolute moves — move in X and Y
m.move_to(x=120, y=150)    # move to center of bed

In [ ]:
# Check current position
m.get_position()

---
## Part 2: Serial Dilution with Syringe
For more details on the syringe tool, see [syringe](./examples/syringe/readme.md) folder for further details.

We'll use the syringe to:
- Fill a 24-well plate with water
- Add a color gradient to column 1
- Serially dilute the color across the remaining columns

Labware layout:
- **Slot 2** — petri dish with colored water
- **Slot 3** — petri dish with water
- **Slot 4** — 24-well plate (sample plate)

### Load the Deck

In [ ]:
deck = m.load_deck('POSE-calibrated-deck.json')

### Load and Pick Up the Syringe

In [ ]:
# Verify tool indices configured on your machine
m.configured_tools

In [ ]:
syringe = Syringe(index=2, name='syringe', config='10cc_syringe_liquidhandling')
m.load_tool(syringe)

In [ ]:
m.pickup_tool(syringe)
m.move_to(z=150)  # drop bed to make room for labware

### Load Labware

See the [labware](./examples/labware/readme.md) folder for further details

Run the cell below, then **physically place** each item in the correct slot on the deck.

In [ ]:
color_dish = m.load_labware('generic_petri_dish_100ml', 2)
water_dish = m.load_labware('generic_petri_dish_100ml', 3)
samples    = m.load_labware('greiner_24_wellplate_3300ul', 4)

# A petri dish has a single well; access it directly for convenience
water = water_dish['A1']
color = color_dish['A1']

### Fill the Plate

In [ ]:
# Fill column 1 with water — leave room for the colored water
# Row A: 20% dye → 2.0 mL water, Row B: 40% → 1.5 mL, Row C: 60% → 1.0 mL, Row D: 80% → 0.5 mL
water_volumes = {'A': 2.0, 'B': 1.5, 'C': 1.0, 'D': 0.5}  # mL

for row, vol in water_volumes.items():
    syringe.transfer(vol=vol, source=water, destination=samples.wells[f'{row}1'])

In [ ]:
# Add colored water to column 1 — total per well = 2.5 mL
# Row A: 20% dye → 0.5 mL, Row B: 40% → 1.0 mL, Row C: 60% → 1.5 mL, Row D: 80% → 2.0 mL
color_volumes = {'A': 0.5, 'B': 1.0, 'C': 1.5, 'D': 2.0}  # mL

for row, vol in color_volumes.items():
    syringe.transfer(
        vol=vol,
        source=color,
        destination=samples.wells[f'{row}1'],
        mix_after=(1, 2)
    )

### Run the Serial Dilution

Transfers 1.5 mL from each column into the next, mixing after each transfer.

In [ ]:
# Serial dilution: transfer 1.25 mL from each column to the next, across ALL rows
# 1.25 mL transfer into 1.25 mL already in the next well → 2x dilution per step
# (each destination well gets 1.25 mL water pre-fill + 1.25 mL transferred = 2.5 mL total)
transfer_volume = 1.25  # mL

# First, pre-fill columns 2-6 with 1.25 mL water in every row
for row in ['A', 'B', 'C', 'D']:
    for column in range(2, 7):
        syringe.transfer(
            vol=1.25,
            source=water,
            destination=samples.wells[f'{row}{column}']
        )

# Now do the serial dilution across columns for each row
for row in ['A', 'B', 'C', 'D']:
    for column in range(1, 6):
        syringe.transfer(
            vol=transfer_volume,
            source=samples.wells[f'{row}{column}'],
            destination=samples.wells[f'{row}{column + 1}'],
            mix_after=(1, 2)
        )

In [ ]:
m.park_tool()     # park the syringe before switching tools
m.move_to(z=150)  # drop bed for clearance

---
## Part 3: Precise Transfer with Pipette

See the [pipette](./examples/pipette/readme.md) folder for further details

The syringe is great for bulk liquid handling, but for precise, small-volume transfers we use an OpenTrons pipette.

We'll transfer **200 µL** from well **A3** (the last well of the serial dilution) into well **A4**.

Make sure the following labware is physically installed on the deck:
- **Slot 0** — pipette tiprack (loaded with tips)
- **Slot 5** — petri dish acting as tip trash

In [ ]:
# Load the tiprack and trash into their deck slots
tiprack = m.load_labware('opentrons_96_tiprack_300ul', 0)   # Uncomment if you have a P300 pipette, otherwise delete!
# tiprack = m.load_labware('opentrons_96_tiprack_1000ul', 0)  # Uncomment if you have a P1000 pipette, otherwise delete!
trash   = m.load_labware('generic_petri_dish_100ml', 5)

In [ ]:
# Load the pipette (tool index 3)
# config_name = "P300"  # Uncomment if you have a P300 pipette, otherwise delete! 
# config_name = "P1000" # Uncomment if you have a P1000 pipette, otherwise delete!

pipette = Pipette.from_config(index=3, name='Pipette', config_file=config_name)
m.load_tool(pipette)

# Associate the tiprack and trash with the pipette
pipette.add_tiprack(tiprack)
pipette.trash = trash[0]   # petri dish is a single well

In [ ]:
m.pickup_tool(pipette)

In [ ]:
# Pick up a fresh tip from the tiprack
pipette.pickup_tip()

In [ ]:
# Aspirate 200 µL from well A3 (last well of the serial dilution), 3 mm above the bottom
pipette.aspirate(200, samples['A3'].bottom(3))

In [ ]:
# Dispense into well A4, just 1 mm inside the top of the well
pipette.dispense(200, samples['A4'].top(-1))

In [ ]:
# Drop the used tip into the trash petri dish
pipette.drop_tip()

In [ ]:
m.park_tool()
m.move_to(z=150)

---
## Part 4: Take a Picture with the Camera

See the [camera](./examples/camera/readme.md) folder for further details

We'll move the camera to well **A1** — the most concentrated color from the dilution series — and take a photo.

In [ ]:
cam = Camera.from_config(1, 'Camera', 'camera_opencv.json')

m.load_tool(cam)
m.pickup_tool(cam)

In [ ]:
# Move to well A1 using its labware coordinates
target_well = samples['A1']
m.move_to(x=target_well.x, y=target_well.y, z=50) 

In [ ]:

# show what the camera sees
cam.show_image();

# save to disk
cam.capture('well_A1.jpg')

In [ ]:
m.park_tool()
m.move_to(z=150)

---
## Part 5: Spectral Measurement
See the [spectral sensor](./examples/spectral-sensor/readme.md) folder for further details

We'll pick up the AS7341 spectral sensor, position it over the same well **A1**, take a measurement, and print the results directly.

In [ ]:
spec = AS7341(index=0, name='AS7341 spectral sensor', config='AS7341')
m.load_tool(spec)

In [ ]:
m.pickup_tool(spec)

`connect()` initiate communication with the Seeed Xiao microcontroller. `blink(time)` turns on the onboard LED so you can visually confirm connection.

In [ ]:
spec.connect()
import time
spec.blink(0.5)  # blink the onboard LED to confirm connection
time.sleep(1)
spec.blink(.5)
time.sleep(1)
spec.blink(.5)

In [ ]:
# Position the sensor over well A1
m.move_to(x=target_well.x, y=target_well.y, z=100)

In [ ]:
# Danger! We can get closer to the well but be careful not to crash into the labware or break the sensor!
m.move_to(z=0)  # drop the sensor down into the well for measurement

In [ ]:
# Turn on the AS7341's onboard white LED and take a reading
# This mirrors the CircuitPython adafruit_as7341 API directly
spec.led_current = 10
spec.led = True
readings = spec.all_channels
spec.led = False

print('Spectral measurement for well A1:')
print(readings)

In [ ]:
m.park_tool()
